# Climate-finance analysis

## Purpose and scope

This notebook keeps two different exercises separate. The first applies a stipulated 2% allocation rate to annual GCW. It describes a scenario rule, not observed finance and not an estimated funding need. The second compares reported forest-capacity losses with published country action-cost curves from Busch et al.; its coverage is partial and the action-cost curve concerns reforestation options rather than a direct price for protecting existing forests.

A 2% allocation rate is not the 2% discount rate used in the separate capitalisation scenario. The notebook displays both inputs explicitly and does not combine them into one quantity.

## 1. Set up and load the current finance outputs

The main allocation table is the source for the 2% GCW scenario. The Busch supplementary files report coverage and the country-level action-cost matching. The assumptions table distinguishes allocation and discount rates.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = RESULTS / 'validation'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo


## 2. Inspect the GCW allocation scenario

The table reports the assigned amount by group under the stipulated rule. The calculation applies the allocation percentage to annual GCW. It does not estimate what a country should contribute, what it receives, or the amount of climate finance actually mobilised.

In [2]:
allocation = pd.read_csv(TABLES / 'Table4_Ibrahim_2pct_allocation.csv')
assumptions = pd.read_csv(TABLES / 'Supplementary_deforestation_scenarios.csv')
accounts = pd.read_csv(TABLES / 'Supplementary_country_level_accounts_2024.csv')

# Reconstruct the stipulated allocation from annual GCW and compare group sums with Table 4.
allocation_rate = 0.02
gcw = pd.to_numeric(accounts['gcw_usd2015_yr_mean'], errors='coerce')
exclude = accounts['exclude_from_country_sum'].fillna(False).astype(bool)
eligible = gcw.notna() & ~exclude
accounts.loc[eligible, 'reconstructed_allocation_usd2015_yr'] = allocation_rate * gcw.loc[eligible]
reconstructed = accounts.loc[eligible].groupby('groups', dropna=False)['reconstructed_allocation_usd2015_yr'].sum().div(1e9).rename('reconstructed_billion_usd2015').reset_index()
comparison = allocation.merge(reconstructed, left_on='group', right_on='groups', how='outer')
comparison['difference_billion_usd2015'] = comparison['reconstructed_billion_usd2015'] - comparison['annual_allocation_billion_usd2015']
display(comparison)
display(assumptions.loc[assumptions['metric'].astype(str).str.contains('rate|allocation|discount', case=False, na=False)])


C:\Users\aurel\AppData\Local\Temp\ipykernel_16976\3376425001.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  accounts.loc[eligible, 'reconstructed_allocation_usd2015_yr'] = allocation_rate * gcw.loc[eligible]


,group,units,annual_allocation_billion_usd2015,share_of_total_pct,median_allocation_per_unit_million_usd2015,groups,reconstructed_billion_usd2015,difference_billion_usd2015
0,Developed economies,62,72.239007,43.720574,113.724556,Developed economies,72.239007,8.795382e-10
1,Developing economies,80,68.601312,41.518964,156.314013,Developing economies,68.601312,-7.365813e-09
2,LDCs,37,21.089087,12.763561,185.188891,LDCs,21.089087,-3.881979e-10
3,SIDS,57,3.299455,1.996900,1.259196,SIDS,3.299455,-3.763221e-10
4,Total,236,165.228860,100.000000,NaN,NaN,NaN,NaN
5,Unclassified,0,0.000000,0.000000,NaN,NaN,NaN,NaN


,metric,value,unit,assumption_or_formula,discount_rate,ibrahim_allocation_rate
4,Perpetuity value of annual lost GCW at 2% disc...,0.986699,trillion US$2015,Annual lost GCW value / 0.02; perpetual consta...,0.02,NaN


## 3. Inspect the Busch action-cost coverage

The coverage table shows how many units have a published curve and how many targets fall within its available supply at the evaluated price brackets. The associated country table carries the matched target, curve availability and cost ranges. A missing or out-of-range curve is not treated as a zero cost.

In [3]:
coverage = pd.read_csv(TABLES / 'Supplementary_Busch_action_cost_coverage.csv')
busch = pd.read_csv(TABLES / 'Supplementary_Busch_action_cost_by_unit.csv')
display(coverage)
display(busch.head(20))
print('Busch output columns:', busch.columns.tolist())

,status,units,reported_loss_capacity_tco2_yr,target_discounted_30yr_tco2,share_of_all_reported_loss_pct,category
0,no_country_curve_in_published_table,45,1.387711e+07,2.133251e+08,39.349667,No country curve in published table
1,target_exceeds_published_supply_at_100,14,4.171775e+06,6.413040e+07,11.829407,Target exceeds published supply at $100/tCO2
2,target_within_positive_price_curve,12,1.499698e+06,2.305404e+07,4.252517,Target within positive-price range
3,target_within_supply_at_nonpositive_price,18,1.055328e+07,1.622297e+08,29.924679,Above -$20 and at or below $0/tCO2
4,target_within_supply_at_or_below_minus20,16,5.164278e+06,7.938761e+07,14.643730,At or below -$20/tCO2


,UNIT_CODE,ISO3,name,region,iso3,groups,reported_loss_capacity_tco2_yr,target_discounted_30yr_tco2,busch_curve_available,source_quantity_at_minus20_tco2_pv,source_quantity_at_zero_tco2_pv,source_quantity_at_100_tco2_pv,marginal_price_lower_usd2020_tco2,marginal_price_upper_usd2020_tco2,status,annualized_uniform_price_payment_lower_usd2020_yr_partial,annualized_uniform_price_payment_upper_usd2020_yr_partial,annualized_uniform_price_payment_lower_usd2015_yr_partial,annualized_uniform_price_payment_upper_usd2015_yr_partial
0,BRA,BRA,Brazil,Americas,BRA,Developing economies,7.860908e+06,1.208414e+08,False,NaN,NaN,NaN,NaN,NaN,no_country_curve_in_published_table,NaN,NaN,NaN,NaN
1,CHN,CHN,"China (incl. Taiwan, Hong Kong SAR and Macao SAR)",Asia,CHN,Developing economies,1.593247e+06,2.449211e+07,True,1680573.0,112000000.0,4.780000e+09,NaN,0.0,target_within_supply_at_nonpositive_price,NaN,NaN,NaN,NaN
2,CAN,CAN,Canada,Americas,CAN,Developed economies,1.532859e+05,2.356380e+06,False,NaN,NaN,NaN,NaN,NaN,no_country_curve_in_published_table,NaN,NaN,NaN,NaN
3,AUS,AUS,Australia,Oceania,AUS,Developed economies,2.162169e+06,3.323784e+07,False,NaN,NaN,NaN,NaN,NaN,no_country_curve_in_published_table,NaN,NaN,NaN,NaN
4,COD,COD,Democratic Republic of the Congo,Africa,COD,LDCs,2.210384e+06,3.397902e+07,True,46000000.0,194000000.0,8.380000e+08,-20.0,-20.0,target_within_supply_at_or_below_minus20,NaN,NaN,NaN,NaN
5,IDN,IDN,Indonesia,Asia,IDN,Developing economies,1.853758e+06,2.849680e+07,True,11000000.0,65000000.0,1.100000e+09,NaN,0.0,target_within_supply_at_nonpositive_price,NaN,NaN,NaN,NaN
6,IND,IND,India,Asia,IND,Developing economies,2.509565e+06,3.857816e+07,True,8716857.0,109000000.0,1.500000e+09,NaN,0.0,target_within_supply_at_nonpositive_price,NaN,NaN,NaN,NaN
7,MEX,MEX,Mexico,Americas,MEX,Developing economies,6.735283e+05,1.035378e+07,True,73000000.0,701000000.0,1.910000e+09,-20.0,-20.0,target_within_supply_at_or_below_minus20,NaN,NaN,NaN,NaN
8,AGO,AGO,Angola,Africa,AGO,LDCs,1.748833e+06,2.688385e+07,True,868254.6,5831668.0,1.790000e+07,NaN,NaN,target_exceeds_published_supply_at_100,NaN,NaN,NaN,NaN
9,COL,COL,Colombia,Americas,COL,Developing economies,4.917179e+05,7.558910e+06,True,81200000.0,223000000.0,1.240000e+09,-20.0,-20.0,target_within_supply_at_or_below_minus20,NaN,NaN,NaN,NaN


Busch output columns: ['UNIT_CODE', 'ISO3', 'name', 'region', 'iso3', 'groups', 'reported_loss_capacity_tco2_yr', 'target_discounted_30yr_tco2', 'busch_curve_available', 'source_quantity_at_minus20_tco2_pv', 'source_quantity_at_zero_tco2_pv', 'source_quantity_at_100_tco2_pv', 'marginal_price_lower_usd2020_tco2', 'marginal_price_upper_usd2020_tco2', 'status', 'annualized_uniform_price_payment_lower_usd2020_yr_partial', 'annualized_uniform_price_payment_upper_usd2020_yr_partial', 'annualized_uniform_price_payment_lower_usd2015_yr_partial', 'annualized_uniform_price_payment_upper_usd2015_yr_partial']


## 4. Compare the scenarios without conflating their units

This compact view places the annual allocation total next to the partial Busch comparison summary. Currency-year, annualisation, time horizon and included country coverage are not interchangeable; use each output's unit fields and source documentation when describing it.

In [4]:
comparison_path = VALIDATION / 'finance_action_cost_comparison.json'
if comparison_path.exists():
    import json
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    display(pd.Series(comparison, name='value').to_frame())
else:
    print('The validation JSON is not present; the table exports above remain available.')

,value
gibbs_gmw_gcw_usd2015_yr,8261443009142.151367
ibrahim_fixed_share,0.02
ibrahim_2pct_proxy_usd2015_yr,165228860182.843018
ibrahim_100bn_30yr_pledge_usd_cumulative,3000000000000.0
ibrahim_published_country_unit_mismatch,annual GCW minus 30-year cumulative allocation...
busch_article,"Busch et al. (2024), 30-year marginal abatemen..."
busch_cost_curve,least-cost of natural regeneration vs plantati...
discount_factor_years_1_to_30,15.372451
positive_loss_reporters,105
positive_loss_total_tco2_yr,35266134.806994


## 5. Questions to check before drafting

- Is the 2% GCW rule labelled as a scenario rather than observed finance or need?
- Are the allocation rate and discount rate reported separately?
- Is the Busch result described only for units and targets covered by the published curves?
- Are the curve price year and the paper's monetary year conversion stated?

These are traceability prompts; final choices about which results enter the manuscript remain with the author.

## 6. Figure 4 data: uncertainty, concentration, and Busch prices

This section regenerates the Figure 4 inputs from the 236-unit allocation table, the global GCW monetary draws, the 105 positive-loss cases, and the published Busch marginal-abatement curves. The 2% rule remains an assumed share of GCW; its P05-P95 range propagates the GCW monetary draws only, with fixed forest rates and areas. It is not a confidence interval for the policy rule.

For each Busch country curve, the 30-year discounted target uses a 5% annuity factor and is compared with cumulative modeled mitigation on the published −20 to 100 US$2020/tCO₂ price grid. Marginal prices inside the grid use piecewise-linear interpolation; values at or below the −20 boundary remain censored. For the 12 positive-price cases only, the exploratory area beneath the curve from Q(0) to the target is shown as a **partial positive-price component**, both in 2020-US$ present value and as a 2015-US$/year annuity. It excludes the negative-price portion, cases beyond the grid, countries without a curve, implementation constraints, and any claim to measure the cost of replacing the forest service.

In [5]:
import json
import subprocess
import sys

subprocess.run([sys.executable, str(ROOT / 'code/scripts/build_finance_figure4.py')],
               cwd=ROOT, check=True, capture_output=True, text=True)
figure4 = json.loads((VALIDATION / 'finance_figure4_summary.json').read_text(encoding='utf-8'))
national = pd.read_csv(TABLES / 'Supplementary_Figure4_finance_allocations_by_unit.csv')
groups = pd.read_csv(TABLES / 'Supplementary_Figure4_finance_allocations_by_group.csv')
prices = pd.read_csv(TABLES / 'Supplementary_Figure4_Busch_marginal_prices_by_unit.csv')
display(pd.DataFrame([{
    'allocation_total_billion_usd2015_yr': figure4['allocation_total_billion_usd2015_yr'],
    'allocation_p05_billion_usd2015_yr': figure4['allocation_uncertainty']['p05_billion_usd2015_yr'],
    'allocation_p95_billion_usd2015_yr': figure4['allocation_uncertainty']['p95_billion_usd2015_yr'],
    'top10_share_pct': figure4['top10_allocation_share_pct'],
    'top50_share_pct': figure4['top50_allocation_share_pct'],
    'busch_within_grid_units': figure4['busch_supply_sufficient_by_100_units'],
    'busch_within_grid_loss_share_pct': figure4['busch_supply_sufficient_by_100_loss_share_pct'],
    'busch_partial_positive_cost_annual_usd2015': figure4['busch_positive_price_partial_integral_annual_usd2015'],
}]))
display(groups)
display(prices.groupby('status', dropna=False).agg(
    units=('iso3', 'size'),
    loss_tco2_yr=('reported_loss_capacity_tco2_yr', 'sum'),
    median_interpolated_price_usd2020_tco2=('marginal_price_interpolated_usd2020_tco2', 'median'),
).reset_index())
display(prices.loc[prices.status.eq('target_within_positive_price_curve'),
                   ['iso3', 'name', 'marginal_price_interpolated_usd2020_tco2',
                    'positive_price_integral_annual_usd2015_partial']])

,allocation_total_billion_usd2015_yr,allocation_p05_billion_usd2015_yr,allocation_p95_billion_usd2015_yr,top10_share_pct,top50_share_pct,busch_within_grid_units,busch_within_grid_loss_share_pct,busch_partial_positive_cost_annual_usd2015
0,165.22886,64.53012,299.983976,64.056074,90.630345,46,48.820927,1.284588e+06


,group_display,units,total_usd2015_yr,median_usd2015_yr,p25_usd2015_yr,p75_usd2015_yr
0,Developed economies,62,7.223901e+10,1.137246e+08,1.910088e+06,3.978624e+08
1,Developing economies,80,6.860131e+10,1.563140e+08,1.696690e+07,6.038059e+08
2,LDCs,37,2.108909e+10,1.851889e+08,5.429996e+07,4.998457e+08
3,SIDS,57,3.299455e+09,1.259196e+06,3.822834e+05,1.426757e+07


,status,units,loss_tco2_yr,median_interpolated_price_usd2020_tco2
0,no_country_curve_in_published_table,45,1.387711e+07,NaN
1,target_exceeds_published_supply_at_100,14,4.171775e+06,NaN
2,target_within_positive_price_curve,12,1.499698e+06,5.432455
3,target_within_supply_at_nonpositive_price,18,1.055328e+07,-12.348572
4,target_within_supply_at_or_below_minus20,16,5.164278e+06,NaN


,iso3,name,marginal_price_interpolated_usd2020_tco2,positive_price_integral_annual_usd2015_partial
20,GAB,Gabon,0.794828,12808.243872
23,CMR,Cameroon,0.962738,179981.620127
29,SUR,Suriname,21.778604,468161.125952
55,PAK,Pakistan,1.114872,47506.566457
61,CIV,Côte d'Ivoire,3.355884,202252.067293
71,GNQ,Equatorial Guinea,8.986944,165535.087285
77,AZE,Azerbaijan,2.804996,191032.722324
89,VUT,Vanuatu,0.339411,489.503002
96,LBY,Libya,7.509026,9500.142943
99,DMA,Dominica,15.605738,1920.583637
